# Bestelldaten mit DXF-Geometrie verknüpfen

Dieses Notebook verbindet die Kundenbestellungen (`customers_sales.csv`) mit den Geometriewerten der DXF-Dateien (Ordner `10991360_editet`) und exportiert das Ergebnis nach Excel.

## Ablauf im Überblick

| Schritt | Baustein | Liefert |
|:-:|---|---|
| 1 | DXF-Dateien auswerten | `df` (eine Zeile pro DXF) |
| 2 | Bestelldaten einlesen | `orders` (eine Zeile pro Order) |
| 2a | Namen auf Schreibfehler prüfen | `orders` (Namen vereinheitlicht), `name_fixes`, `name_review_df` |
| 3 | Bestelldaten validieren | Konsolen-Report (ändert nichts) |
| 4 | DXF-Werte validieren | Konsolen-Report und `invalid_geo` |
| 5 | Bereinigen mit Protokoll | `orders_clean`, `df_clean`, `removed_report`, `dup_dxf`, `suspicious` |
| 6 | Zusammenführen | `merged` (eine Zeile pro Order mit DXF-Werten) |
| 7 | Kontrolle und Export | Excel-Datei mit 3 Sheets |

**Wichtig:** Die Zellen bauen aufeinander auf und müssen **in dieser Reihenfolge** laufen. Nach Änderungen an `helpers.py` den Kernel neu starten, da Jupyter importierte Module im Speicher behält.

In [28]:
import re
from difflib import SequenceMatcher
import pandas as pd
import os
from helpers import *
import openpyxl

## 1. DXF-Dateien auswerten

**Was passiert:** Alle `.dxf`-Dateien im Ordner `10991360_editet` werden nacheinander geöffnet. Für jede Datei werden vier Kennzahlen berechnet und als Zeile gesammelt.

**Baut auf:**
- `helpers.py` (`get_dxf_bounds`, `get_cutting_length`, `get_number_of_holes`)
- dem Ordner `10991360_editet` mit den DXF-Dateien

**Liefert:** `df` mit einer Zeile pro DXF und den Spalten

| Spalte | Bedeutung |
|---|---|
| `file` | Dateiname, z. B. `Geo (10).DXF` (späterer Join-Schlüssel) |
| `bounding_box_width` / `_height` | Breite und Höhe der umschließenden Box |
| `cutting_length` | Gesamte Schnittlänge |
| `num_holes` | Anzahl der Löcher (Innenkonturen) |

**Hinweise:**
- Dateien, die nicht auf `.dxf` enden (Groß-/Kleinschreibung egal), werden übersprungen.
- Schlägt eine Berechnung fehl, liefert `helpers.py` `None`. Das wird in `df` zu `NaN` und in Schritt 4 sichtbar.
- Die Excel-Ausgabe von `df` ist auskommentiert, `df` wird nur im Speicher weiterverwendet.

In [29]:

path = "10991360_editet"

rows = []

for file in os.listdir(path):
    file_path = os.path.join(path, file)

    # Optional: process only DXF files
    if not file.lower().endswith(".dxf"):
        continue

    bounding_box_width, bounding_box_height = get_dxf_bounds(file_path)
    cutting_length = get_cutting_length(file_path)
    num_holes = get_number_of_holes(file_path)

    rows.append({
        "file": file,
        "bounding_box_width": bounding_box_width,
        "bounding_box_height": bounding_box_height,
        "cutting_length": cutting_length,
        "num_holes": num_holes,
    })

df = pd.DataFrame(rows)
# df.to_excel("ergebnisse.xlsx", index=False, engine="openpyxl")

## 2. Bestelldaten einlesen

**Was passiert:** Die CSV wird gelesen und die Spalten werden in saubere Datentypen gebracht.
- `sep=None, engine="python"` erkennt das Trennzeichen (Tab, `;` oder `,`) automatisch.
- Spaltennamen und Textspalten werden von Leerzeichen am Rand befreit, damit der spätere Vergleich nicht an unsichtbaren Leerzeichen scheitert.
- `order_quantity` wird in eine Zahl umgewandelt. Was nicht lesbar ist (z. B. `"1.600"` oder `"n/a"`), wird zu `NaN` (`errors="coerce"`).

**Baut auf:** `customers_sales.csv` mit den Spalten `customer_name`, `customer_branch`, `file_name`, `order_quantity`

**Liefert:** `orders` mit einer Zeile pro Bestellung. Es wird nichts gelöscht, die Zeilenzahl entspricht der CSV.

In [30]:
# Bestelldaten einlesen
orders = pd.read_csv("customers_sales.csv", sep=None, engine="python")
orders.columns = orders.columns.str.strip()

for col in ["customer_name", "customer_branch", "file_name"]:
    orders[col] = orders[col].astype("string").str.strip()
orders["order_quantity"] = pd.to_numeric(orders["order_quantity"], errors="coerce")

# print(orders.shape)
# orders.head()

## 2a. Schreibfehler in Kunden- und Filialnamen erkennen und vereinheitlichen

**Was passiert:** Ähnlich geschriebene Namen (z. B. `AutoPart AG` und `AutoParts AG`) werden erkannt und der **häufigsten Schreibweise** zugeordnet. Das passiert **vor** der Validierung, damit Duplikatprüfung und Merge mit einheitlichen Namen arbeiten.

- **Ähnlichkeit:** `difflib.SequenceMatcher` (0 bis 1) auf einer vereinfachten Schreibweise (Kleinbuchstaben, ohne Satzzeichen und überzählige Leerzeichen).
- **Ab `SIM_THRESHOLD` (0,90)** wird automatisch zugeordnet. **Zwischen `REVIEW_THRESHOLD` (0,80) und 0,90** wird nur gemeldet, aber nichts geändert.
- **Filialen** werden nur **innerhalb desselben Kunden** verglichen (`Frankfurt Branch` bei zwei verschiedenen Kunden ist erlaubt).
- **Schutzregeln gegen Fehlzuordnungen:**
  - Unterschiedliche **Zahlen** (`Filiale 1` / `Filiale 2`) werden nie zusammengeführt.
  - Unterschiedliche **Rechtsform** (`MetalForm AG` / `MetalForm SA`) werden nie automatisch zusammengeführt, sondern nur zur Prüfung gemeldet, da es sich um verschiedene Firmen handeln kann.
- **Eigene Vorgaben:** In `MANUAL_MAP` kannst du Zuordnungen festlegen, z. B. `{"AutoPart AG": "AutoParts AG"}`. Mit `{"X": "X"}` verhinderst du, dass `X` jemals zusammengeführt wird.
- **`APPLY = False`:** nur berichten, `orders` bleibt unverändert. Empfehlung: erst so laufen lassen, Bericht prüfen, dann auf `True` setzen.

**Baut auf:** `orders` aus Schritt 2

**Liefert:**
- `orders` mit korrigierten Namen (bei `APPLY = True`). Die ursprünglichen Schreibweisen bleiben in `customer_name_original` und `customer_branch_original` erhalten.
- `name_fixes`: alle Korrekturen (Original, Korrektur, Anzahl Zeilen, Ähnlichkeit)
- `name_review_df`: ähnliche Namen, die **nicht** zusammengeführt wurden und geprüft werden sollten

**Hinweise:**
- Die Zuordnung ist eine Heuristik. Sehr kurze Namen oder sehr ähnliche echte Firmennamen können falsch gruppiert werden. Deshalb die Korrekturliste vor der Weiterverarbeitung ansehen.
- Nach der Vereinheitlichung können in Schritt 3 und 5 **mehr exakte Duplikate** erkannt werden als vorher, weil sich vormals verschieden geschriebene Zeilen jetzt gleichen. Die Originalschreibweise steht in `removed_report` und `suspicious` daneben.
- Nicht geprüft werden `file_name`-Tippfehler. Orders ohne passende DXF zeigt Schritt 6.

In [31]:
# Schreibfehler in Kunden-/Filialnamen erkennen und vereinheitlichen
# ---------- Einstellungen ----------
SIM_THRESHOLD = 0.90      # ab dieser Ähnlichkeit wird automatisch zugeordnet
REVIEW_THRESHOLD = 0.80   # ab dieser Ähnlichkeit (aber unter SIM_THRESHOLD) nur melden
APPLY = True              # False = nur berichten, orders bleibt unverändert
MANUAL_MAP = {}           # Ausnahmen/Vorgaben, z. B. {"AutoPart AG": "AutoParts AG"}
                          # oder {"MetalForm SA": "MetalForm SA"} = nie zusammenführen


# Rechtsformen: Unterschiedliche Rechtsform (AG vs. SA) = vermutlich andere Firma, nie automatisch zusammenführen
LEGAL_FORMS = {"ag", "sa", "gmbh"}


def _norm(s):
    """Klein, ohne Satzzeichen/Mehrfach-Leerzeichen, damit nur echte Tippfehler zählen."""
    return re.sub(r"[^\w]+", " ", str(s).lower()).strip()


def _similar(a, b):
    na, nb = _norm(a), _norm(b)
    # Unterschiedliche Zahlen (z. B. "Filiale 1" vs. "Filiale 2") sind nie derselbe Wert
    if re.findall(r"\d+", na) != re.findall(r"\d+", nb):
        return 0.0
    ratio = SequenceMatcher(None, na, nb).ratio()
    if {w for w in na.split() if w in LEGAL_FORMS} != {w for w in nb.split() if w in LEGAL_FORMS}:
        return min(ratio, SIM_THRESHOLD - 0.001)   # höchstens "bitte prüfen", nie automatisch
    return ratio


def build_mapping(values):
    """Ordnet seltene Schreibweisen der häufigsten ähnlichen Schreibweise zu."""
    counts = values.dropna().value_counts()
    order = sorted(counts.index, key=lambda v: (-counts[v], v))   # häufigste zuerst
    canon, mapping, review = [], {}, []
    for v in order:
        if v in MANUAL_MAP:
            target = MANUAL_MAP[v]
            if target not in canon:
                canon.append(target)
            mapping[v] = target
            continue
        best, best_sim = None, 0.0
        for c in canon:
            s = _similar(v, c)
            if s > best_sim:
                best, best_sim = c, s
        if best is not None and best_sim >= SIM_THRESHOLD:
            mapping[v] = best
        else:
            canon.append(v)
            mapping[v] = v
            if best is not None and best_sim >= REVIEW_THRESHOLD:
                review.append((v, best, round(best_sim, 3)))
    return mapping, counts, review


fix_rows, review_rows = [], []

# ---------- 1) customer_name ----------
orders["customer_name_original"] = orders["customer_name"]
name_map, name_counts, name_review = build_mapping(orders["customer_name"])
for old, new in name_map.items():
    if old != new:
        fix_rows.append({"feld": "customer_name", "kunde": new, "original": old, "korrigiert": new,
                         "anzahl_zeilen": int(name_counts[old]),
                         "aehnlichkeit": round(_similar(old, new), 3)})
for a, b, s in name_review:
    review_rows.append({"feld": "customer_name", "kunde": "-", "wert_a": a, "wert_b": b, "aehnlichkeit": s})
customer_final = orders["customer_name"].map(name_map).astype("string")

# ---------- 2) customer_branch (nur innerhalb desselben Kunden vergleichen) ----------
orders["customer_branch_original"] = orders["customer_branch"]
branch_final = orders["customer_branch"].copy()
for cust, grp in orders.groupby(customer_final, dropna=True):
    b_map, b_counts, b_review = build_mapping(grp["customer_branch"])
    branch_final.loc[grp.index] = grp["customer_branch"].map(b_map)
    for old, new in b_map.items():
        if old != new:
            fix_rows.append({"feld": "customer_branch", "kunde": cust, "original": old, "korrigiert": new,
                             "anzahl_zeilen": int(b_counts[old]),
                             "aehnlichkeit": round(_similar(old, new), 3)})
    for a, b, s in b_review:
        review_rows.append({"feld": "customer_branch", "kunde": cust, "wert_a": a, "wert_b": b, "aehnlichkeit": s})

name_fixes = pd.DataFrame(fix_rows, columns=["feld", "kunde", "original", "korrigiert", "anzahl_zeilen", "aehnlichkeit"])
name_review_df = pd.DataFrame(review_rows, columns=["feld", "kunde", "wert_a", "wert_b", "aehnlichkeit"])

# ---------- 3) Anwenden + Bericht ----------
if APPLY:
    orders["customer_name"] = customer_final
    orders["customer_branch"] = branch_final.astype("string")

print(f"Korrekturen {'ANGEWENDET' if APPLY else 'nur vorgeschlagen (APPLY=False)'}: {len(name_fixes)} Schreibweisen "
      f"({int(name_fixes['anzahl_zeilen'].sum()) if len(name_fixes) else 0} Zeilen betroffen)")
if len(name_fixes):
    print(name_fixes.to_string(index=False))
print(f"\nÄhnlich, aber NICHT zusammengeführt (bitte prüfen): {len(name_review_df)}")
if len(name_review_df):
    print(name_review_df.to_string(index=False))

Korrekturen ANGEWENDET: 2 Schreibweisen (4 Zeilen betroffen)
         feld        kunde     original   korrigiert  anzahl_zeilen  aehnlichkeit
customer_name AutoParts AG  AutoPart AG AutoParts AG              3         0.957
customer_name MetalForm SA Metalform SA MetalForm SA              1         1.000

Ähnlich, aber NICHT zusammengeführt (bitte prüfen): 0


## 3. Bestelldaten validieren (nur Prüfung)

**Was passiert:** Die Bestellungen werden auf Auffälligkeiten untersucht und die Anzahl je Problem ausgegeben. **Es wird nichts verändert oder gelöscht.**

| Prüfung | Bedeutung |
|---|---|
| Fehlende Werte | leere Felder je Spalte |
| Ungültige Mengen | `NaN`, ≤ 0 oder mit Nachkommastelle |
| Exakte Duplikate | alle 4 Spalten identisch (`keep=False` markiert **alle** Beteiligten, auch das Original) |
| Gleiche Order, andere Menge | gleicher Kunde, gleiche Filiale und gleiche Datei, aber andere Menge. Das kann eine Wiederholungsbestellung sein und wird deshalb nur gemeldet |

**Baut auf:** `orders` aus Schritt 2

**Liefert:** Konsolen-Ausgabe. Zusätzlich entstehen `cols` (die 4 Prüfspalten, werden in Schritt 5 wiederverwendet) sowie `invalid_qty`, `exact_dups` und `same_order_diff_qty` zum Ansehen im Detail.

In [32]:
# Bestelldaten validieren
cols = ["customer_name", "customer_branch", "file_name", "order_quantity"]

print("Fehlende Werte:\n", orders[cols].isna().sum(), "\n")

invalid_qty = orders[
    orders["order_quantity"].isna()
    | (orders["order_quantity"] <= 0)
    | (orders["order_quantity"] % 1 != 0)
]
print("Ungültige Mengen:", len(invalid_qty))

exact_dups = orders[orders.duplicated(subset=cols, keep=False)]
print("Exakte Duplikate:", len(exact_dups))

# gleiche Kunde/Filiale/Datei, aber andere Menge (nur zur Prüfung, wird nicht gelöscht)
key = ["customer_name", "customer_branch", "file_name"]
same_order_diff_qty = orders[
    orders.duplicated(subset=key, keep=False) & ~orders.duplicated(subset=cols, keep=False)
]
print("Gleiche Order, andere Menge:", len(same_order_diff_qty))

Fehlende Werte:
 customer_name      0
customer_branch    0
file_name          0
order_quantity     0
dtype: int64 

Ungültige Mengen: 0
Exakte Duplikate: 8
Gleiche Order, andere Menge: 0


## 4. DXF-Werte validieren (nur Prüfung)

**Was passiert:** Die berechneten DXF-Werte werden auf Plausibilität geprüft: Dateinamen eindeutig? Fehlende Werte? Breite, Höhe oder Schnittlänge ≤ 0?

Der Dateiname in `df` wird dabei von Leerzeichen befreit (wie in Schritt 2 bei den Bestellungen), damit der Join in Schritt 6 zuverlässig greift.

**Baut auf:** `df` aus Schritt 1

**Liefert:**
- Konsolen-Report
- `geo_cols` (die drei Geometriespalten, werden in Schritt 5 und 6 wiederverwendet)
- `invalid_geo`: die betroffenen Zeilen, als Tabelle direkt unter der Zelle angezeigt

**Hinweis:** `num_holes = 0` gilt als gültig (Teil ohne Löcher). Deshalb wird `num_holes` nur auf fehlende Werte geprüft, nicht auf ≤ 0.

In [33]:
# DXF-Werte validieren
geo_cols = ["bounding_box_width", "bounding_box_height", "cutting_length"]

df["file"] = df["file"].astype("string").str.strip()

print("DXF-Dateien:", len(df))
print("Doppelte Dateinamen:", df["file"].duplicated().sum())
print("Fehlende Werte:\n", df[geo_cols + ["num_holes"]].isna().sum(), "\n")

invalid_geo = df[df[geo_cols].isna().any(axis=1) | (df[geo_cols] <= 0).any(axis=1)]
print("Fehlerhafte Geometriewerte:", len(invalid_geo))
invalid_geo

DXF-Dateien: 80
Doppelte Dateinamen: 0
Fehlende Werte:
 bounding_box_width     0
bounding_box_height    0
cutting_length         0
num_holes              0
dtype: int64 

Fehlerhafte Geometriewerte: 0


,file,bounding_box_width,bounding_box_height,cutting_length,num_holes


## 5. Bereinigen mit Protokoll

**Was passiert:** Hier wird tatsächlich bereinigt, jede Entfernung wird dabei festgehalten.

1. **Exakte Duplikate entfernen:** Gelöscht wird **nur**, wenn Kunde, Filiale, Datei **und** Menge identisch sind. Die jeweils erste Zeile bleibt. `gleich_wie_index` zeigt, zu welcher Zeile das Duplikat gehört.
2. **Auffällige Zeilen markieren, nicht löschen:** Fehlende Werte, Mengen ≤ 0 oder mit Nachkommastelle bleiben erhalten und landen in `suspicious`. Der Rohtext der Menge aus der CSV steht daneben (`rohwert_menge`), um zu sehen, was dort wirklich stand.
3. **DXF-Werte bereinigen:** Doppelte Dateinamen werden entfernt (erste Zeile bleibt). Ungültige Geometriewerte (`NaN` oder ≤ 0) werden auf `NaN` gesetzt und aufgelistet, die Zeile bleibt bestehen.

**Baut auf:** `orders` (Schritt 2), `cols` (Schritt 3), `df` und `geo_cols` (Schritt 4), außerdem der CSV, die hier nochmals als Rohtext (`dtype=str`) gelesen wird

**Liefert:**

| Variable | Inhalt |
|---|---|
| `orders_clean` | Bestellungen ohne exakte Duplikate (Basis für den Merge) |
| `removed_report` | entfernte Orders mit Grund und `gleich_wie_index` |
| `suspicious` | behaltene, aber auffällige Orders |
| `df_clean` | DXF-Werte ohne doppelte Dateinamen, ungültige Werte als `NaN` |
| `dup_dxf` | entfernte doppelte DXF-Zeilen |

**Hinweise:**
- `raw` und `orders` müssen denselben Index haben. Das ist der Fall, solange in Schritt 2 keine Zeilen gefiltert werden.
- `order_quantity` wird nur in ganze Zahlen (`int`) umgewandelt, wenn es keine auffälligen Zeilen gibt. Sonst bleibt der Typ erhalten, weil `NaN` kein `int` sein kann.
- Zwei Zeilen mit identisch leeren Feldern zählen als Duplikat.

In [34]:
# Rohwerte der CSV (als Text), damit man bei nicht lesbaren Mengen sieht, was dort stand
raw = pd.read_csv("customers_sales.csv", sep=None, engine="python", dtype=str)
raw.columns = raw.columns.str.strip()

# --- 1) Gedroppt werden NUR exakte Duplikate (alle 4 Spalten gleich) ---
dup_mask = orders.duplicated(subset=cols, keep="first")

# Index der ersten Zeile, mit der das Duplikat übereinstimmt
first_idx = orders.assign(_idx=orders.index).groupby(cols, dropna=False)["_idx"].transform("min")

removed_report = orders[dup_mask].copy()
removed_report["grund"] = "Exaktes Duplikat (alle 4 Spalten gleich)"
removed_report["gleich_wie_index"] = first_idx[dup_mask]

print(f"Orders: {len(orders)} -> {len(orders) - len(removed_report)} "
      f"({len(removed_report)} entfernt)\n")
if len(removed_report):
    print("Entfernte Zeilen:")
    print(removed_report.to_string(), "\n")

orders_clean = orders[~dup_mask].copy()

# --- 2) Auffällige Zeilen: werden NICHT entfernt, nur angezeigt ---
suspicious_mask = (
    orders_clean[cols].isna().any(axis=1)
    | (orders_clean["order_quantity"] <= 0)
    | (orders_clean["order_quantity"] % 1 != 0)
)
suspicious = orders_clean[suspicious_mask].copy()
suspicious["rohwert_menge"] = raw.loc[suspicious.index, "order_quantity"]

print(f"Auffällige Zeilen (BEHALTEN, bitte prüfen): {len(suspicious)}")
if len(suspicious):
    print(suspicious.to_string())

# Menge nur in int umwandeln, wenn alle Werte gültig sind
if not suspicious_mask.any():
    orders_clean["order_quantity"] = orders_clean["order_quantity"].astype(int)

# --- 3) DXF-Werte: nur doppelte Dateinamen entfernen, ungültige Werte auf NaN setzen ---
dup_dxf = df[df.duplicated(subset="file", keep="first")]
print(f"\nEntfernte doppelte DXF-Zeilen: {len(dup_dxf)}")
if len(dup_dxf):
    print(dup_dxf.to_string())

df_clean = df.drop_duplicates(subset="file").copy()
for c in geo_cols:
    bad = ~(df_clean[c] > 0)
    if bad.any():
        print(f"\nUngültige Werte in {c} -> NaN gesetzt:")
        print(df_clean.loc[bad, ["file", c]].to_string())
    df_clean.loc[bad, c] = pd.NA

Orders: 79 -> 75 (4 entfernt)

Entfernte Zeilen:
        customer_name   customer_branch     file_name  order_quantity customer_name_original customer_branch_original                                     grund  gleich_wie_index
7   PrecisionCut GmbH  Stuttgart Branch  Geo (15).DXF            4000      PrecisionCut GmbH         Stuttgart Branch  Exaktes Duplikat (alle 4 Spalten gleich)                 3
49       AutoParts AG     Munich Branch  Geo (55).DXF            9250           AutoParts AG            Munich Branch  Exaktes Duplikat (alle 4 Spalten gleich)                42
62  PrecisionCut GmbH  Stuttgart Branch  Geo (56).DXF            4800      PrecisionCut GmbH         Stuttgart Branch  Exaktes Duplikat (alle 4 Spalten gleich)                50
68       AutoParts AG     Berlin Branch  Geo (12).DXF            8800           AutoParts AG            Berlin Branch  Exaktes Duplikat (alle 4 Spalten gleich)                 4 

Auffällige Zeilen (BEHALTEN, bitte prüfen): 0

Entfernte do

## 6. Zusammenführen (Merge)

**Was passiert:** Die DXF-Werte werden an die Bestellungen angehängt. Verknüpft wird `file_name` (Bestellung) mit `file` (DXF).

- `how="left"`: **Jede Bestellung bleibt erhalten**, auch ohne passende DXF (dann `NaN` in den DXF-Spalten).
- `validate="many_to_one"`: Mehrere Bestellungen dürfen dieselbe DXF nutzen, jede DXF darf aber nur einmal in `df_clean` vorkommen. Sonst bricht pandas mit Fehler ab, statt stillschweigend Zeilen zu vervielfachen.
- `indicator=True`: Hilfsspalte `_merge`, die zeigt, ob eine Zeile gefunden wurde (`both`) oder nicht (`left_only`). Sie wird nach der Auswertung wieder entfernt.
- Die doppelte Schlüsselspalte `file` wird gelöscht.

**Baut auf:** `orders_clean` und `df_clean` aus Schritt 5, `geo_cols` aus Schritt 4

**Liefert:** `merged` mit einer Zeile pro Order und den Spalten der Bestellung plus `bounding_box_width`, `bounding_box_height`, `cutting_length`, `num_holes`. Die Konsolen-Ausgabe zeigt, wie viele Orders ohne DXF geblieben sind und wie viele Zeilen fehlende DXF-Werte haben.

**Hinweis:** Passt ein Dateiname nicht exakt (Tippfehler oder `.dxf` statt `.DXF`), bleibt die Zeile ohne DXF-Werte. Der Vergleich unterscheidet Groß- und Kleinschreibung.

In [35]:
# Zusammenführen
merged = orders_clean.merge(
    df_clean,
    left_on="file_name",
    right_on="file",
    how="left",
    validate="many_to_one",   # mehrere Orders pro DXF erlaubt, jede DXF nur einmal
    indicator=True,
).drop(columns="file")

print(merged["_merge"].value_counts())
print("Orders ohne DXF:", (merged["_merge"] == "left_only").sum())
print("Zeilen mit fehlenden DXF-Werten:", merged[geo_cols].isna().any(axis=1).sum())

merged = merged.drop(columns="_merge")
merged.head()

_merge
both          75
left_only      0
right_only     0
Name: count, dtype: int64
Orders ohne DXF: 0
Zeilen mit fehlenden DXF-Werten: 0


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,cutting_length,num_holes
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,3438.024977,0
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,3217.207505,5
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,1742.517642,7
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,1302.774283,2
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,2764.783512,11


## 7. Kontrolle und Export nach Excel

**Was passiert:**
1. **Info:** Zählt DXF-Dateien, zu denen keine Bestellung existiert (nur Anzeige).
2. **Sicherheitsprüfung:** Der `assert` bricht ab, wenn der Merge die Zeilenzahl verändert hat. Er vergleicht mit `orders_clean`, Verluste aus Schritt 5 siehst du also nur im dortigen Protokoll.
3. **Protokolltabellen** für die Excel vorbereiten (`original_index` ist der Zeilenindex im DataFrame, bei 0 beginnend, die CSV-Zeile ist etwa Index + 2).
4. **Export** in `customers_sales_with_dxf_data.xlsx` mit fünf Sheets, danach werden die Spaltenbreiten automatisch angepasst (maximal 60 Zeichen).

**Baut auf:** `merged` (Schritt 6), `orders_clean` und `df_clean` (Schritt 5), `removed_report` und `dup_dxf` (Schritt 5), `name_fixes` und `name_review_df` (Schritt 2a)

**Liefert:** die Excel-Datei `customers_sales_with_dxf_data.xlsx`

| Sheet | Inhalt |
|---|---|
| `Daten` | `merged`, die fertige Haupttabelle |
| `Entfernte Orders` | gedroppte Duplikate mit Grund |
| `Entfernte DXF` | doppelte DXF-Zeilen mit Grund |
| `Namenskorrekturen` | alle vereinheitlichten Schreibweisen aus Schritt 2a |
| `Namen zu prüfen` | ähnliche, aber nicht zusammengeführte Namen aus Schritt 2a |

**Hinweise:**
- Sheets mit 0 entfernten Zeilen bleiben bestehen (nur mit Kopfzeile). So siehst du, dass geprüft wurde.
- Die Datei wird bei jedem Lauf überschrieben. Ist sie in Excel geöffnet, schlägt das Schreiben fehl.
- Die Zelle endet mit `merged`, sodass die Tabelle auch im Notebook angezeigt wird.

In [36]:
# DXF-Dateien ohne Bestellung (nur zur Info)
unused = set(df_clean["file"]) - set(orders_clean["file_name"])
print("DXF ohne Bestellung:", len(unused))

assert len(merged) == len(orders_clean), "Merge hat Zeilen verändert!"

# ---------- Tabellen der entfernten Zeilen vorbereiten ----------
orders_drop = removed_report.reset_index().rename(columns={"index": "original_index"})

dxf_drop = dup_dxf.reset_index().rename(columns={"index": "original_index"})
dxf_drop["grund"] = "Doppelter Dateiname (erste Zeile behalten)"

# ---------- Export ----------
with pd.ExcelWriter("customers_sales_with_dxf_data.xlsx", engine="openpyxl") as writer:
    merged.to_excel(writer, sheet_name="Daten", index=False)
    orders_drop.to_excel(writer, sheet_name="Entfernte Orders", index=False)
    dxf_drop.to_excel(writer, sheet_name="Entfernte DXF", index=False)
    name_fixes.to_excel(writer, sheet_name="Namenskorrekturen", index=False)
    name_review_df.to_excel(writer, sheet_name="Namen zu prüfen", index=False)

    # Spaltenbreiten automatisch anpassen
    for ws in writer.sheets.values():
        for column_cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in column_cells)
            ws.column_dimensions[column_cells[0].column_letter].width = min(width + 2, 60)

print(f"Export fertig: customers_sales_with_dxf_data.xlsx"
      f"({len(merged)} Orders, {len(orders_drop)} entfernte Orders, {len(dxf_drop)} entfernte DXF-Zeilen, "
      f"{len(name_fixes)} Namenskorrekturen)")
merged

DXF ohne Bestellung: 5
Export fertig: customers_sales_with_dxf_data.xlsx(75 Orders, 4 entfernte Orders, 0 entfernte DXF-Zeilen, 2 Namenskorrekturen)


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,cutting_length,num_holes
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,3438.024977,0
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,3217.207505,5
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,1742.517642,7
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,1302.774283,2
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,2764.783512,11
...,...,...,...,...,...,...,...,...,...,...
70,AutoParts AG,Munich Branch,Geo (78).DXF,8150,AutoParts AG,Munich Branch,386.361049,290.000000,5581.874650,52
71,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,PrecisionCut GmbH,Stuttgart Branch,36.553638,36.553638,147.032435,4
72,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,TechFab Industries,Hamburg Branch,801.705747,344.180571,6111.748725,15
73,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,MetalForm SA,Frankfurt Branch,1968.901094,832.434782,16821.059941,15
